# 06. Reranking
## Candidate Documents → Reranker → Final Top-k

### 이번 실습의 위치

```text
01 Naive RAG
    ↓
02 Document Processing
    ↓
03 Query Rewriting
    ↓
04 Multi Query
    ↓
05 Retrieval Engineering
   Dense + BM25 + RRF
    ↓
06 Reranking
   Candidate → Reorder → Final Top-k
```

이번 실습의 핵심은 하나이다.

> **Retriever가 가져온 후보 문서를 Reranker가 질문과 다시 비교하여 순서를 재정렬한다.**

### 학습 목표

1. Retriever와 Reranker의 역할 차이를 설명한다.
2. Retriever에서 후보 문서를 넉넉하게 가져온다.
3. Query-Document pair를 만든다.
4. Reranker score로 문서를 다시 정렬한다.
5. 최종 Top-k 문서만 선택한다.


# 1. 왜 Reranking이 필요한가?

Retriever는 많은 문서 중에서 관련 있어 보이는 문서를 **빠르게 찾는 역할**을 한다.

하지만 Retriever의 상위 문서가 항상 질문에 가장 정확한 문서라는 보장은 없다.

```text
Query
  ↓
Retriever
  ↓
Candidate Top-8
  ↓
Reranker
  ↓
Final Top-3
```

따라서 역할을 다음처럼 구분하면 된다.

| 단계 | 역할 |
|---|---|
| Retriever | 관련 문서를 빠르게 후보로 찾는다 |
| Reranker | 후보를 질문과 다시 비교해 순서를 바꾼다 |

05번에서 만든 Hybrid + RRF도 **후보 문서를 만드는 Retriever 단계**이다.

이번 06번에서는 Reranking 자체를 쉽게 이해하기 위해 Dense Retrieval로 후보를 가져온 뒤 재정렬한다.


# 2. 환경 설정

02번에서 만든 ChromaDB를 그대로 사용한다.

Reranker는 `CrossEncoder`를 사용한다.

```text
Embedding Retriever
Query와 Document를 각각 vector로 변환

Cross-Encoder Reranker
Query와 Document를 함께 입력하여 관련성을 계산
```

필요 패키지:

```bash
uv add sentence-transformers
```


In [1]:
import sys
from pathlib import Path

from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder

# 상위 폴더의 common_config.py를 사용할 수 있도록 경로를 추가한다.
sys.path.append("..")

# 02번과 같은 임베딩 모델을 사용한다.
from common_config import embedding_model


# 02번 실습에서 만든 ChromaDB 위치
DB_PATH = Path("chroma_db/ai_trend_report_doc_chunks")
COLLECTION_NAME = "ai_trend_report_2026_doc_chunks"

# ChromaDB가 없으면 02번 실습을 먼저 실행해야 한다.
if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_docling_chunks.ipynb를 먼저 실행하세요."
    )

# Dense Retrieval에 사용할 임베딩 모델
embeddings = embedding_model()

# 기존 ChromaDB 연결
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)

print("저장된 chunk 수:", vector_store._collection.count())


저장된 chunk 수: 48


# 3. Retriever로 후보 문서 가져오기

Reranker는 전체 문서를 처음부터 검색하지 않는다.

먼저 Retriever가 후보를 가져오고, 그 후보만 다시 평가한다.

이번 실습에서는 구조를 단순하게 보기 위해 Dense Retrieval로 **Top-8** 후보를 가져온다.

```text
전체 문서
   ↓
Retriever
   ↓
후보 8개
```

실제 Advanced RAG에서는 이 자리에 05번의 **Hybrid + RRF 결과**를 넣을 수 있다.


In [2]:
# 실습 질문
query = "추론형 데이터는 어떤 유형으로 구분되나요?"

# Retriever가 관련 문서 후보 8개를 가져온다.
candidate_docs = vector_store.similarity_search(
    query,
    k=8,
)

print("[Retriever 후보 문서]")

# 현재 검색 순서를 확인한다.
for rank, doc in enumerate(candidate_docs, start=1):
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


[Retriever 후보 문서]
1. chunk=45 | pages=21
2. chunk=17 | pages=9
3. chunk=15 | pages=8
4. chunk=22 | pages=11
5. chunk=23 | pages=11
6. chunk=18 | pages=9
7. chunk=20 | pages=10
8. chunk=19 | pages=9


# 4. Reranker 모델 불러오기

Reranker는 Query와 Document를 **함께 읽고** 관련성을 계산한다.

```text
(Query, Document 1) → relevance score
(Query, Document 2) → relevance score
(Query, Document 3) → relevance score
...
```

이번 실습에서는 작은 크기의 Reranker 모델을 사용한다.


In [3]:
# Query와 Document의 관련성을 다시 평가할 모델
RERANKER_MODEL = "Qwen/Qwen3-Reranker-0.6B"

# 첫 실행에서는 Hugging Face에서 모델을 다운로드할 수 있다.
reranker = CrossEncoder(
    RERANKER_MODEL
)

print("Reranker 모델:", RERANKER_MODEL)


modules.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Reranker 모델: Qwen/Qwen3-Reranker-0.6B


# 5. Query-Document pair 만들기

Cross-Encoder는 다음 형태로 입력한다.

```text
[
    (Query, Document 1),
    (Query, Document 2),
    (Query, Document 3)
]
```

즉 각 후보 문서가 **현재 질문과 얼마나 관련 있는지** 직접 비교한다.


In [4]:
# 후보 문서마다 같은 Query를 붙여 Query-Document pair를 만든다.
pairs = []

for doc in candidate_docs:
    pair = (
        query,
        doc.page_content,
    )
    pairs.append(pair)

print("Pair 수:", len(pairs))

# 첫 번째 pair를 확인한다.
print("\n[Query]")
print(pairs[0][0])

print("\n[Document]")
print(pairs[0][1][:500])


Pair 수: 8

[Query]
추론형 데이터는 어떤 유형으로 구분되나요?

[Document]
5.2 AI 기술 토픽으로 본 노력 및 대응
< 추론형 데이터 종류(예시) >
① 단계적 '과정' 추론, 설명 = Ÿ AI가 최종 결론에 도달하는 '과정' 자체를 학습하는 데 중점을 둔 데이터 - 절차적 추론 데이터(CoT), 상호작용적 추론 데이터 등 - (예시) 민원처리 절차의 체크리스트 및 결정 기준. ② '근거' 기반 설명, 설명 = Ÿ AI가 답을 말할 때 어떤 근거를 인용해야 하는지 학습시키기 위한 데이터 - 자연어 설명(NLE) 데이터, 추출적 근거 데이터 등 - (예시) 정책 Q&A 공공데이터 개방률 ? → 포털 통계 페이지 2-3 근거를 인용. ③ '맥락·관계' 이해, 설명 = Ÿ 객체 간의 '관계'나 사회적 상호작용의 '숨은 의도' 등 이해를 위한 데이터 - 물리적·시각적 맥락 데이터, 사회적·의도적 맥락 데이터 등 - (예시) 작업지시 최적화 : 부품 교체 전 선행 작업? → 선후관계 그래프 기반 절차 도출. ④ 논리·취약점 검증, 설명 = Ÿ AI 모델의 논리적


# 6. Reranker로 다시 정렬하기

이제 모든 Query-Document pair의 관련성 점수를 계산한다.

점수가 높을수록 현재 질문과 더 관련성이 높은 문서라고 본다.

```text
Retriever 순위
1, 2, 3, 4, 5 ...

        ↓ Reranker

새로운 순위
3, 1, 5, 2, 4 ...
```

Reranking의 핵심은 **새 문서를 찾는 것이 아니라 기존 후보의 순서를 바꾸는 것**이다.


In [5]:
# 각 Query-Document pair의 관련성 점수를 계산한다.
scores = reranker.predict(
    pairs,
    show_progress_bar=True,
)

# Document와 score를 하나씩 묶는다.
doc_scores = list(
    zip(candidate_docs, scores)
)

# 관련성 점수가 높은 순서로 다시 정렬한다.
reranked_docs = sorted(
    doc_scores,
    key=lambda item: float(item[1]),
    reverse=True,
)

print("[Reranker 결과]")

for rank, (doc, score) in enumerate(reranked_docs, start=1):
    print(
        f"{rank}. "
        f"score={float(score):.4f} "
        f"| chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Reranker 결과]
1. score=6.9375 | chunk=45 | pages=21
2. score=-5.5000 | chunk=23 | pages=11
3. score=-6.4062 | chunk=17 | pages=9
4. score=-6.8750 | chunk=15 | pages=8
5. score=-7.1875 | chunk=19 | pages=9
6. score=-7.5000 | chunk=22 | pages=11
7. score=-7.6562 | chunk=20 | pages=10
8. score=-7.7500 | chunk=18 | pages=9


# 7. Retriever 순위와 Reranker 순위 비교

학생들이 확인할 것은 복잡한 평가 지표가 아니다.

다음 두 가지만 보면 된다.

1. Retriever의 1위 문서가 Reranker에서도 1위인가?
2. 아래에 있던 문서가 Reranking 후 위로 올라왔는가?

순위가 바뀌었다면 Reranker가 Query와 Document의 관련성을 다시 평가한 것이다.


In [6]:
# Retriever에서 각 chunk가 몇 위였는지 저장한다.
retriever_rank = {}

for rank, doc in enumerate(candidate_docs, start=1):
    chunk_id = doc.metadata.get("chunk_id")
    retriever_rank[chunk_id] = rank


print("Reranker | Retriever | chunk")
print("-" * 35)

# Reranking 후 순위와 기존 Retriever 순위를 함께 확인한다.
for new_rank, (doc, score) in enumerate(reranked_docs, start=1):
    chunk_id = doc.metadata.get("chunk_id")
    old_rank = retriever_rank.get(chunk_id)

    print(
        f"{new_rank:>8} | "
        f"{old_rank:>9} | "
        f"{chunk_id}"
    )


Reranker | Retriever | chunk
-----------------------------------
       1 |         1 | 45
       2 |         5 | 23
       3 |         2 | 17
       4 |         3 | 15
       5 |         8 | 19
       6 |         4 | 22
       7 |         7 | 20
       8 |         6 | 18


# 8. Final Top-k 선택

Reranking을 하는 이유는 최종적으로 LLM에게 더 관련성 높은 문서만 전달하기 위해서이다.

예를 들어 Retriever가 8개 후보를 찾았다면,
Reranker 후 상위 3개만 최종 Context로 사용할 수 있다.

```text
Retriever Top-8
      ↓
Reranker
      ↓
Final Top-3
      ↓
LLM Context
```


In [7]:
# Reranker 결과 중 상위 3개만 최종 문서로 선택한다.
final_docs = [
    doc
    for doc, score in reranked_docs[:3]
]

print("[Final Top-3]")

for rank, doc in enumerate(final_docs, start=1):
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


[Final Top-3]
1. chunk=45 | pages=21
2. chunk=23 | pages=11
3. chunk=17 | pages=9


# 9. 핵심 정리

## 오늘 기억할 것

```text
Retriever
→ 관련 문서를 빠르게 찾는다.

Reranker
→ 후보 문서를 Query와 다시 비교한다.

Reranking
→ 새로운 문서를 찾는 것이 아니라 순서를 다시 정한다.
```

전체 흐름은 다음과 같다.

```text
Query
  ↓
Retriever
  ↓
Candidate Top-N
  ↓
Query-Document Pair
  ↓
Reranker
  ↓
Final Top-k
```

### 왜 두 단계를 나누는가?

- Reranker는 정확하지만 모든 문서에 적용하면 느리다.
- 따라서 Retriever가 먼저 후보를 줄인다.
- Reranker는 줄어든 후보만 정밀하게 평가한다.

### 05번과 연결

```text
05 Retrieval Engineering
Dense + BM25
      ↓
RRF
      ↓
Candidate Top-N

06 Reranking
Candidate Top-N
      ↓
Cross-Encoder
      ↓
Final Top-k
```

> 05번은 **좋은 후보를 찾는 단계**,  
> 06번은 **후보의 순서를 더 정확하게 정하는 단계**이다.
